In [1]:
import numpy as np
from TextDataset import TextDataset
from NegativeSampler import NegativeSampler
from Word2VecModel import Word2VecModel
from Vocabulary import Vocabulary

In [2]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
text = "\n".join([d["text"] for d in dataset if d["text"].strip()])

/opt/anaconda3/envs/machine-learning/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
with open("dataset.txt","w", encoding="utf-8") as f:
    text = f.write(text)

In [ ]:

with open("dataset.txt","r", encoding="utf-8") as f:
    text = f.read()


In [ ]:
dataset = TextDataset(text,10)
pairs = dataset.generate_pairs(2)
epochs = 10
num_negative_samples = 5
model = Word2VecModel(dataset.vocab_size,30)
sampler = NegativeSampler(dataset.counts)

print(len(pairs))
np.random.shuffle(pairs)
loss = 0.0
counter = 0
# negatives = sampler.sample(num_negative_samples*len(pairs)//50)
# for num,(center,context) in enumerate(pairs):
#     if(num%1000==0):
#         loss += model.compute_loss(model.forward_pass((center,context)),model.words_out[negatives] @ model.words_in[center])
#         counter +=1
        
# loss/=counter
# print(f"Mean initial loss: {loss:.4f}")

1137090


In [8]:
for epoch in range(epochs):
    print(f"Epoch: {epoch+1}")
    np.random.shuffle(pairs)
    negatives = sampler.sample(num_negative_samples*len(pairs))
    for num,(center,context) in enumerate(pairs):
        model.training_step((center,context),negatives[num_negative_samples*num:num_negative_samples*num+num_negative_samples])
    #model.change_learning_rate(0.8)
    

Epoch: 1
Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
Epoch: 6
Epoch: 7
Epoch: 8
Epoch: 9
Epoch: 10


In [18]:
word = input()
d=(dataset.get_idx_by_word(word))
print(word)
print(dataset.get_word_by_idx(model.get_k_closest_neighbours(d,5)))

america
['california', 'sydney', 'finland', 'england', 'canada']


In [19]:
word = input()
d=(dataset.get_idx_by_word(word))
print(word)
print(dataset.get_word_by_idx(model.get_k_closest_neighbours(d,5)))

poland
['queen', 'occupation', 'spanish', 'frederick', 'leonard']


In [66]:
word = input()
d=(dataset.get_idx_by_word(word))
print(word)
print(dataset.get_word_by_idx(model.get_k_closest_neighbours(d,5)))

five
['three', 'four', 'six', 'seven', 'nine']


In [67]:
print(np.std(model.words_in))

0.6429802562132777


In [70]:
model.save("stable_model/model.npz")
dataset.vocabulary.save("stable_model/vocab.pkl")